# Notebook 01 — Data Source Inventory

**Purpose:** inventory local raw and Roboflow working copies, compute checksums and lightweight metadata, identify missing expected sources, and save reproducible evidence. This notebook never authenticates, downloads data, calls the Roboflow API, trains a model, or runs detection/tracking.

## 2. Experiment metadata

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import os
import platform
import sys
import time

import torch

EXPERIMENT_ID = 'NOTEBOOK_01_DATA_SOURCE_INVENTORY'
NOTEBOOK_STARTED_AT = datetime.now(timezone.utc)
NOTEBOOK_START_TIME = time.perf_counter()

def find_project_root(start: Path) -> Path:
    required_markers = ('AGENTS.md', 'FISH_AI_PROJECT_WORKFLOW.md', '.git')
    candidates = (start.resolve(), *start.resolve().parents)
    for candidate in candidates:
        if all((candidate / marker).exists() for marker in required_markers):
            return candidate
    raise RuntimeError(f'Cannot resolve project root from {start.resolve()}')

CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_WORKING_DIRECTORY)
CONDA_ENV = os.environ.get('CONDA_DEFAULT_ENV', 'not set')
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = 'cuda:0' if CUDA_AVAILABLE else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'Not available'

print('Notebook: 01_data_source_inventory')
print(f'Experiment ID: {EXPERIMENT_ID}')
print(f'Datetime: {NOTEBOOK_STARTED_AT.isoformat()}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Current working directory: {CURRENT_WORKING_DIRECTORY}')
print(f'Python executable: {sys.executable}')
print(f'Conda env: {CONDA_ENV}')
print(f'Device: {DEVICE}')
print(f'GPU: {GPU_NAME}')
assert CONDA_ENV == 'fish', 'Select the fish kernel before running this notebook.'

Notebook: 01_data_source_inventory
Experiment ID: NOTEBOOK_01_DATA_SOURCE_INVENTORY
Datetime: 2026-08-17T04:16:21.280702+00:00
PROJECT_ROOT: /home/diy-hus/fish
Current working directory: /home/diy-hus/fish/notebooks
Python executable: /home/diy-hus/miniconda3/envs/fish/bin/python
Conda env: fish
Device: cuda:0
GPU: NVIDIA GeForce RTX 3050


## 3. Resolve PROJECT_ROOT

In [2]:
root_markers = {
    'AGENTS.md': (PROJECT_ROOT / 'AGENTS.md').is_file(),
    'FISH_AI_PROJECT_WORKFLOW.md': (PROJECT_ROOT / 'FISH_AI_PROJECT_WORKFLOW.md').is_file(),
    '.git': (PROJECT_ROOT / '.git').is_dir(),
}
for marker, present in root_markers.items():
    print(f'{marker}: {"FOUND" if present else "MISSING"}')
assert all(root_markers.values()), 'One or more project-root markers are missing.'

AGENTS.md: FOUND
FISH_AI_PROJECT_WORKFLOW.md: FOUND
.git: FOUND


## 4. Load configuration

In [3]:
import yaml

PATHS_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'paths.yaml'
DATA_SOURCES_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'data_sources.yaml'
with PATHS_CONFIG_PATH.open('r', encoding='utf-8') as handle:
    PATHS_CONFIG = yaml.safe_load(handle)
with DATA_SOURCES_CONFIG_PATH.open('r', encoding='utf-8') as handle:
    DATA_SOURCES_CONFIG = yaml.safe_load(handle)

sources = DATA_SOURCES_CONFIG['sources']
raw_config = sources['raw_archive']
roboflow_config = sources['labeled_detection_dataset']
RAW_ROOT = PROJECT_ROOT / raw_config['local_root']
ROBOFLOW_ROOT = PROJECT_ROOT / roboflow_config['local_root']

print('CONFIG')
print(f'Raw source: {raw_config["source_of_truth"]}')
print(f'Raw access: {raw_config["transfer_method"]}')
print(f'Automatic raw download: {raw_config["automatic_download"]}')
print(f'Local raw root: {RAW_ROOT.relative_to(PROJECT_ROOT)}')
print(f'Labeled detection source: {roboflow_config["source_of_truth"]}')
print(f'Automatic Roboflow API access: {roboflow_config["automatic_api_access"]}')
print(f'Local Roboflow root: {ROBOFLOW_ROOT.relative_to(PROJECT_ROOT)}')
print('Expected raw inputs:')
for name, item in raw_config.get('expected_inputs', {}).items():
    print(f'  - {name}: {item["path"]} (required_next={item.get("required_for_next_step", False)})')
print('Expected labeled dataset inputs:')
for name, item in roboflow_config.get('expected_inputs', {}).items():
    print(f'  - {name}: {item["path"]} (version={item.get("dataset_version")}, required_next={item.get("required_for_next_step", False)})')

CONFIG
Raw source: google_drive
Raw access: user_manual_download_or_copy
Automatic raw download: False
Local raw root: data/raw
Labeled detection source: roboflow
Automatic Roboflow API access: False
Local Roboflow root: data/roboflow
Expected raw inputs:
  - front_video: data/raw/front (required_next=False)
  - top_video: data/raw/top (required_next=False)
  - sensor_data: data/raw/sensors (required_next=False)
Expected labeled dataset inputs:
  - front_detection_dataset: data/roboflow/front_detect_v1 (version=None, required_next=True)
  - top_detection_dataset: data/roboflow/top_detect_v1 (version=None, required_next=False)


## 5. Check/create local directory structure

In [4]:
LOCAL_DIRECTORIES = [
    PROJECT_ROOT / 'data',
    PROJECT_ROOT / 'data' / 'raw',
    PROJECT_ROOT / 'data' / 'raw' / 'front',
    PROJECT_ROOT / 'data' / 'raw' / 'top',
    PROJECT_ROOT / 'data' / 'raw' / 'sensors',
    PROJECT_ROOT / 'data' / 'roboflow',
    PROJECT_ROOT / 'data' / 'processed',
    PROJECT_ROOT / 'data' / 'eval',
]
for directory in LOCAL_DIRECTORIES:
    existed_before = directory.is_dir()
    directory.mkdir(parents=True, exist_ok=True)
    status = 'EXISTS' if existed_before else 'CREATED_EMPTY'
    print(f'{directory.relative_to(PROJECT_ROOT)}: {status}')

data: CREATED_EMPTY
data/raw: CREATED_EMPTY
data/raw/front: CREATED_EMPTY
data/raw/top: CREATED_EMPTY
data/raw/sensors: CREATED_EMPTY
data/roboflow: CREATED_EMPTY
data/processed: CREATED_EMPTY
data/eval: CREATED_EMPTY


## 6. Discover files

In [5]:
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv'}
SENSOR_EXTENSIONS = {'.csv', '.tsv'}

def infer_category(path: Path, source_type: str) -> str:
    parts = {part.lower() for part in path.parts}
    if 'front' in parts or any(part.startswith('front_') for part in parts):
        return 'front'
    if 'top' in parts or any(part.startswith('top_') for part in parts):
        return 'top'
    if 'sensor' in parts or 'sensors' in parts or path.suffix.lower() in SENSOR_EXTENSIONS:
        return 'sensor'
    if source_type == 'roboflow':
        return 'dataset'
    return 'other'

def required_for_next_step(path: Path, source_config: dict) -> bool:
    for expected in source_config.get('expected_inputs', {}).values():
        expected_path = (PROJECT_ROOT / expected['path']).resolve()
        try:
            path.resolve().relative_to(expected_path)
        except ValueError:
            continue
        return bool(expected.get('required_for_next_step', False))
    return False

discovered_files = []
source_roots = (('raw', RAW_ROOT, raw_config), ('roboflow', ROBOFLOW_ROOT, roboflow_config))
for source_type, root, source_config in source_roots:
    source_files = sorted(path for path in root.rglob('*') if path.is_file())
    print(f'Discovered {len(source_files)} files under {root.relative_to(PROJECT_ROOT)}')
    for path in source_files:
        discovered_files.append({
            'source_type': source_type,
            'source_of_truth': source_config['source_of_truth'],
            'category': infer_category(path.relative_to(root), source_type),
            'required_for_next_step': required_for_next_step(path, source_config),
            'path': path,
        })
print(f'Total discovered files: {len(discovered_files)}')

Discovered 0 files under data/raw
Discovered 0 files under data/roboflow
Total discovered files: 0


## 7. Compute checksums

In [6]:
import hashlib

HASH_CHUNK_SIZE = 8 * 1024 * 1024
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(HASH_CHUNK_SIZE), b''):
            digest.update(chunk)
    return digest.hexdigest()

total_files = len(discovered_files)
for index, item in enumerate(discovered_files, start=1):
    relative_path = item['path'].relative_to(PROJECT_ROOT)
    print(f'Hashing {index}/{total_files}: {relative_path}')
    item['sha256'] = sha256_file(item['path'])

## 8. Read video and sensor metadata

In [7]:
import csv
import cv2
import pandas as pd

metadata_warnings = []
for item in discovered_files:
    path = item['path']
    suffix = path.suffix.lower()
    item.update({
        'video_readable': None, 'fps': None, 'frame_count': None,
        'duration_sec': None, 'width': None, 'height': None,
        'sensor_columns': None, 'sensor_row_count': None,
    })
    if suffix in VIDEO_EXTENSIONS:
        capture = cv2.VideoCapture(str(path))
        try:
            readable = capture.isOpened()
            item['video_readable'] = bool(readable)
            if readable:
                fps = float(capture.get(cv2.CAP_PROP_FPS))
                frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
                item['fps'] = fps
                item['frame_count'] = frame_count
                item['duration_sec'] = frame_count / fps if fps > 0 else None
                item['width'] = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
                item['height'] = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
            else:
                warning = f'Unreadable video: {path.relative_to(PROJECT_ROOT)}'
                metadata_warnings.append(warning)
                print(f'WARNING: {warning}')
        except (OSError, ValueError, cv2.error) as exc:
            item['video_readable'] = False
            warning = f'Video metadata error for {path.relative_to(PROJECT_ROOT)}: {type(exc).__name__}: {exc}'
            metadata_warnings.append(warning)
            print(f'WARNING: {warning}')
        finally:
            capture.release()
    elif suffix in SENSOR_EXTENSIONS:
        try:
            separator = '\t' if suffix == '.tsv' else ','
            item['sensor_columns'] = list(pd.read_csv(path, sep=separator, nrows=0).columns)
            if path.stat().st_size <= 10 * 1024 * 1024:
                with path.open('r', encoding='utf-8', errors='replace', newline='') as handle:
                    item['sensor_row_count'] = max(sum(1 for _ in csv.reader(handle, delimiter=separator)) - 1, 0)
        except (OSError, UnicodeError, pd.errors.ParserError) as exc:
            warning = f'Sensor metadata warning for {path.relative_to(PROJECT_ROOT)}: {type(exc).__name__}: {exc}'
            metadata_warnings.append(warning)
            print(f'WARNING: {warning}')
print(f'Video/sensor metadata warnings: {len(metadata_warnings)}')

Video/sensor metadata warnings: 0


## 9. Inspect local Roboflow structure

In [8]:
roboflow_data_yaml_files = sorted(ROBOFLOW_ROOT.rglob('data.yaml'))
roboflow_dataset_reports = []
for data_yaml_path in roboflow_data_yaml_files:
    with data_yaml_path.open('r', encoding='utf-8') as handle:
        dataset_config = yaml.safe_load(handle) or {}
    dataset_root = data_yaml_path.parent
    split_status = {name: (dataset_root / name).exists() for name in ('train', 'valid', 'test')}
    report = {
        'data_yaml': str(data_yaml_path.relative_to(PROJECT_ROOT)),
        'names': dataset_config.get('names'),
        'nc': dataset_config.get('nc'),
        'train': dataset_config.get('train'),
        'val': dataset_config.get('val'),
        'test': dataset_config.get('test'),
        'split_directories': split_status,
        'images_directories': len(list(dataset_root.rglob('images'))),
        'labels_directories': len(list(dataset_root.rglob('labels'))),
    }
    roboflow_dataset_reports.append(report)
    print(yaml.safe_dump(report, sort_keys=False, allow_unicode=True))
ROBOFLOW_DATASET_LOCAL = 'DETECTED' if roboflow_dataset_reports else 'MISSING'
print(f'ROBOFLOW_DATASET_LOCAL = {ROBOFLOW_DATASET_LOCAL}')

ROBOFLOW_DATASET_LOCAL = MISSING


## 10. Build inventory table

In [9]:
INVENTORY_COLUMNS = [
    'source_type', 'source_of_truth', 'category', 'relative_path', 'exists',
    'required_for_next_step', 'filename', 'extension',
    'file_size_bytes', 'file_size_mb', 'sha256', 'video_readable', 'fps',
    'frame_count', 'duration_sec', 'width', 'height', 'sensor_columns',
    'sensor_row_count',
]
inventory_rows = []
for item in discovered_files:
    path = item['path']
    size_bytes = path.stat().st_size
    inventory_rows.append({
        'source_type': item['source_type'],
        'source_of_truth': item['source_of_truth'],
        'category': item['category'],
        'relative_path': str(path.relative_to(PROJECT_ROOT)),
        'exists': True,
        'required_for_next_step': item['required_for_next_step'],
        'filename': path.name,
        'extension': path.suffix.lower(),
        'file_size_bytes': size_bytes,
        'file_size_mb': round(size_bytes / (1024 ** 2), 4),
        'sha256': item['sha256'],
        'video_readable': item['video_readable'],
        'fps': item['fps'],
        'frame_count': item['frame_count'],
        'duration_sec': item['duration_sec'],
        'width': item['width'],
        'height': item['height'],
        'sensor_columns': item['sensor_columns'],
        'sensor_row_count': item['sensor_row_count'],
    })
inventory = pd.DataFrame(inventory_rows, columns=INVENTORY_COLUMNS)
print(f'Inventory rows: {len(inventory)}')
display(inventory.head(20))

Inventory rows: 0


,source_type,source_of_truth,category,relative_path,exists,required_for_next_step,filename,extension,file_size_bytes,file_size_mb,sha256,video_readable,fps,frame_count,duration_sec,width,height,sensor_columns,sensor_row_count


## 11. Save evidence

In [10]:
LOG_DATA_DIR = PROJECT_ROOT / 'logs' / 'data'
LOG_DATA_DIR.mkdir(parents=True, exist_ok=True)
INVENTORY_PATH = LOG_DATA_DIR / 'raw_inventory.csv'
SUMMARY_PATH = LOG_DATA_DIR / 'notebook01_summary.txt'
inventory.to_csv(INVENTORY_PATH, index=False)
print(f'Output path: {INVENTORY_PATH.relative_to(PROJECT_ROOT)}')
print(f'Records: {len(inventory)}')
print(f'File size: {INVENTORY_PATH.stat().st_size} bytes')

Output path: logs/data/raw_inventory.csv
Records: 0
File size: 225 bytes


## 12. Final summary

In [11]:
expected_sources = {}
for source_name, source_config in sources.items():
    for input_name, input_config in source_config.get('expected_inputs', {}).items():
        input_path = PROJECT_ROOT / input_config['path']
        expected_sources[f'{source_name}.{input_name}'] = {
            'path': input_config['path'],
            'exists': input_path.exists() and any(input_path.rglob('*')) if input_path.is_dir() else input_path.exists(),
            'required_for_next_step': bool(input_config.get('required_for_next_step', False)),
        }
missing_expected_sources = [name for name, state in expected_sources.items() if not state['exists']]
video_rows = inventory[inventory['extension'].isin(VIDEO_EXTENSIONS)]
raw_front_videos = inventory[(inventory['source_type'] == 'raw') & (inventory['category'] == 'front') & (inventory['extension'].isin(VIDEO_EXTENSIONS))]
raw_top_videos = inventory[(inventory['source_type'] == 'raw') & (inventory['category'] == 'top') & (inventory['extension'].isin(VIDEO_EXTENSIONS))]
sensor_files = inventory[inventory['category'] == 'sensor']
roboflow_files = inventory[inventory['source_type'] == 'roboflow']
runtime_sec = time.perf_counter() - NOTEBOOK_START_TIME
warnings = list(metadata_warnings)
if len(raw_front_videos) + len(raw_top_videos) == 0:
    warnings.append('No local raw video was found.')
if ROBOFLOW_DATASET_LOCAL == 'MISSING':
    warnings.append('No local Roboflow YOLO dataset was detected.')
if missing_expected_sources:
    warnings.append(f'Missing expected sources: {missing_expected_sources}')
checkpoint_result = 'PASS_WITH_WARNING' if warnings else 'PASS'
summary = {
    'datetime': datetime.now(timezone.utc).isoformat(),
    'experiment_id': EXPERIMENT_ID,
    'project_root': str(PROJECT_ROOT),
    'Python': sys.executable,
    'Conda env': CONDA_ENV,
    'device': DEVICE,
    'GPU': GPU_NAME,
    'runtime_sec': round(runtime_sec, 3),
    'total_files': len(inventory),
    'total_size_bytes': int(inventory['file_size_bytes'].sum()) if len(inventory) else 0,
    'raw_front_videos': len(raw_front_videos),
    'raw_top_videos': len(raw_top_videos),
    'sensor_files': len(sensor_files),
    'roboflow_files': len(roboflow_files),
    'video_readable': int((video_rows['video_readable'] == True).sum()),
    'video_unreadable': int((video_rows['video_readable'] == False).sum()),
    'local_roboflow_dataset': ROBOFLOW_DATASET_LOCAL,
    'missing_expected_sources': missing_expected_sources,
    'checkpoint_result': checkpoint_result,
    'output_files': [str(INVENTORY_PATH.relative_to(PROJECT_ROOT)), str(SUMMARY_PATH.relative_to(PROJECT_ROOT))],
    'warnings': warnings,
    'next_step': 'User reviews this inventory; Notebook 02 may be prepared only after explicit PASS confirmation.',
}
SUMMARY_PATH.write_text(''.join(f'{key}: {value}\n' for key, value in summary.items()), encoding='utf-8')
print('FINAL SUMMARY')
for key, value in summary.items():
    print(f'{key}: {value}')
print(f'Summary output: {SUMMARY_PATH.relative_to(PROJECT_ROOT)}')
print(f'Summary file size: {SUMMARY_PATH.stat().st_size} bytes')
display(pd.DataFrame([summary]))

FINAL SUMMARY
datetime: 2026-08-17T04:16:22.041468+00:00
experiment_id: NOTEBOOK_01_DATA_SOURCE_INVENTORY
project_root: /home/diy-hus/fish
Python: /home/diy-hus/miniconda3/envs/fish/bin/python
Conda env: fish
device: cuda:0
GPU: NVIDIA GeForce RTX 3050
runtime_sec: 0.761
total_files: 0
total_size_bytes: 0
raw_front_videos: 0
raw_top_videos: 0
sensor_files: 0
roboflow_files: 0
video_readable: 0
video_unreadable: 0
local_roboflow_dataset: MISSING
missing_expected_sources: ['raw_archive.front_video', 'raw_archive.top_video', 'raw_archive.sensor_data', 'labeled_detection_dataset.front_detection_dataset', 'labeled_detection_dataset.top_detection_dataset']
checkpoint_result: PASS_WITH_WARNING
output_files: ['logs/data/raw_inventory.csv', 'logs/data/notebook01_summary.txt']
warnings: ['No local raw video was found.', 'No local Roboflow YOLO dataset was detected.', "Missing expected sources: ['raw_archive.front_video', 'raw_archive.top_video', 'raw_archive.sensor_data', 'labeled_detection_data

,datetime,experiment_id,project_root,Python,Conda env,device,GPU,runtime_sec,total_files,total_size_bytes,...,sensor_files,roboflow_files,video_readable,video_unreadable,local_roboflow_dataset,missing_expected_sources,checkpoint_result,output_files,warnings,next_step
0,2026-08-17T04:16:22.041468+00:00,NOTEBOOK_01_DATA_SOURCE_INVENTORY,/home/diy-hus/fish,/home/diy-hus/miniconda3/envs/fish/bin/python,fish,cuda:0,NVIDIA GeForce RTX 3050,0.761,0,0,...,0,0,0,0,MISSING,"[raw_archive.front_video, raw_archive.top_vide...",PASS_WITH_WARNING,"[logs/data/raw_inventory.csv, logs/data/notebo...","[No local raw video was found., No local Robof...",User reviews this inventory; Notebook 02 may b...
